<a href="https://colab.research.google.com/github/Flamers-Team/Techchalleng3/blob/main/notebooks/conexao.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🔌 Conexão — subir o Assistente Médico no Colab

Este notebook liga todas as peças do projeto e abre a interface web:

1. GPU + clonar o repositório
2. Instalar dependências
3. Trazer o **modelo fine-tuned** do Google Drive
4. Construir o **índice RAG** (ChatBulário)
5. Subir a **interface Gradio** (link público)

> **Antes de começar:** menu **Runtime → Change runtime type → GPU** (T4 já serve).
> Rode as células **na ordem**, de cima para baixo.


---
## 1. GPU + clonar o repositório

Ao abrir pelo botão *"Open in Colab"*, o Colab carrega **só este `.ipynb`** — o resto
do projeto não vem junto. A célula abaixo clona o repositório e entra na pasta dele.

Se o repositório for **privado**, gere um *Personal Access Token* no GitHub
(Settings → Developer settings → Tokens) e cole em `GITHUB_TOKEN`.


In [ ]:
import os

GITHUB_TOKEN = ""   # deixe "" se o repo for público
REPO_DIR = "/content/Techchalleng3"
REPO_URL = "https://github.com/Flamers-Team/Techchalleng3.git"

# GPU
gpu = os.popen("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader").read().strip()
print("GPU:", gpu or "❌ NENHUMA — ative em Runtime → Change runtime type → GPU")

# clonar (ou atualizar se já existe)
url = REPO_URL.replace("https://", f"https://{GITHUB_TOKEN}@") if GITHUB_TOKEN else REPO_URL
if not os.path.isdir(REPO_DIR):
    os.system(f"git clone -q {url} {REPO_DIR}")
    print("✅ repositório clonado")
else:
    os.system(f"git -C {REPO_DIR} pull -q")
    print("✅ repositório atualizado (git pull)")

os.chdir(REPO_DIR)
print("Pasta atual:", os.getcwd())
print("Conteúdo   :", sorted(os.listdir())[:12], "...")


---
## 2. Instalar dependências  (~3–5 min)

Instala o Unsloth (para carregar o modelo), o ChromaDB + sentence-transformers
(para o RAG) e o Gradio + ReportLab (para a interface e os PDFs).

⚠️ **Se o Colab pedir para reiniciar a sessão** ("Restart runtime"), reinicie e
rode **de novo a partir da Seção 1** (a Seção 1 é idempotente).


In [ ]:
# Unsloth cuida da stack de treino/inferência (torch, transformers, peft, bitsandbytes...)
%pip install -q --upgrade unsloth unsloth_zoo
# libs da aplicação (RAG + UI + PDF)
%pip install -q "chromadb>=0.5.5" "sentence-transformers>=3.0" "gradio==4.44.0" reportlab datasets

# --- checagem rápida ---
import importlib, sys
faltou = []
for m in ["torch", "unsloth", "transformers", "peft", "bitsandbytes",
          "chromadb", "sentence_transformers", "gradio", "reportlab", "datasets"]:
    try:
        mod = importlib.import_module(m)
        print(f"  ✅ {m:22s} {getattr(mod, '__version__', '?')}")
    except Exception as e:
        faltou.append(m); print(f"  ❌ {m:22s} {e}")

import torch
print("\nCUDA disponível:", torch.cuda.is_available())
if faltou:
    print("\n⚠️  Alguns imports falharam. Menu Runtime → Restart runtime e rode de novo a partir da Seção 1.")


---
## 3. Modelo fine-tuned (Google Drive)

O modelo é o **adapter LoRA** (~164 MB) que você salvou no Drive em
`techchallenge_fase3/biomistral-medquad-lora/`.

A célula monta o Drive e copia a pasta para a **raiz do repositório** com o nome
`biomistral-medquad-lora` — é exatamente onde `src/llm/client.py` procura o modelo
por padrão, então o resto do projeto passa a usar o modelo **real** (não o mock).


In [ ]:
import os, json, shutil
from google.colab import drive

drive.mount("/content/drive")

DRIVE_MODELO = "/content/drive/MyDrive/techchallenge_fase3/biomistral-medquad-lora"
DST_MODELO   = "/content/Techchalleng3/biomistral-medquad-lora"   # raiz do repo

assert os.path.isdir(DRIVE_MODELO), (
    f"❌ não encontrei o modelo em {DRIVE_MODELO}\n"
    "   confira o caminho no seu Drive (Seção 9 do notebook de fine-tuning)."
)

if os.path.isdir(DST_MODELO):
    shutil.rmtree(DST_MODELO)              # evita erro em re-execução
shutil.copytree(DRIVE_MODELO, DST_MODELO)
print("✅ modelo copiado para", DST_MODELO)
print("   arquivos:", sorted(os.listdir(DST_MODELO)))

# garante que o adapter aponte para o modelo base no HuggingFace (e não um caminho local)
cfg_path = os.path.join(DST_MODELO, "adapter_config.json")
cfg = json.load(open(cfg_path))
base = str(cfg.get("base_model_name_or_path", ""))
if not base.startswith("BioMistral/"):
    cfg["base_model_name_or_path"] = "BioMistral/BioMistral-7B"
    json.dump(cfg, open(cfg_path, "w"), indent=2)
    print(f"   adapter_config.json: base_model corrigido ('{base}' → 'BioMistral/BioMistral-7B')")
else:
    print("   adapter_config.json: base_model OK →", base)


---
## 4. Índice RAG (ChatBulário)

O RAG usa o dataset **ChatBulário** (bulas de medicamentos em PT-BR). São 3 arquivos
`.jsonl` que **não estão no Git** (grandes). A célula tenta pegá-los, nesta ordem:

1. do seu Google Drive (`techchallenge_fase3/data/`), se existirem lá;
2. senão, baixa direto do dataset `walmeidadf/chat_bulario` no HuggingFace.

Depois roda `build_index_chatbulario.py`, que cria o ChromaDB em
`data/processed/chroma_index/`.

> Indexar **8.000** documentos leva ~4–5 min (CPU). Para indexar **tudo (~69k)**,
> troque `8000` por nada na última linha — leva ~35 min.


In [ ]:
import os, shutil

RAW = "/content/Techchalleng3/data/raw"
DRIVE_DATA = "/content/drive/MyDrive/techchallenge_fase3/data"   # ajuste se estiver noutro lugar
os.makedirs(RAW, exist_ok=True)

arquivos = ["chatbulario_train.jsonl", "chatbulario_validation.jsonl", "chatbulario_test.jsonl"]
faltando = [f for f in arquivos if not os.path.exists(f"{RAW}/{f}")]

# 1) tenta copiar do Drive
for f in list(faltando):
    for cand in (f"{DRIVE_DATA}/{f}", f"{DRIVE_DATA}/raw/{f}"):
        if os.path.exists(cand):
            shutil.copy(cand, f"{RAW}/{f}")
            faltando.remove(f)
            print(f"  ✅ {f} (copiado do Drive)")
            break

# 2) o que faltar, baixa do HuggingFace
if faltando:
    print("  ⬇️  baixando do HuggingFace: walmeidadf/chat_bulario ...")
    from datasets import load_dataset
    ds = load_dataset("walmeidadf/chat_bulario")
    mapa = {"train": "chatbulario_train.jsonl",
            "validation": "chatbulario_validation.jsonl",
            "test": "chatbulario_test.jsonl"}
    for split, fname in mapa.items():
        if fname in faltando and split in ds:
            ds[split].to_json(f"{RAW}/{fname}", force_ascii=False)
            print(f"  ✅ {fname} (do HuggingFace)")

print("\nArquivos prontos:", [f for f in arquivos if os.path.exists(f"{RAW}/{f}")])


In [ ]:
# constrói o índice (8000 docs = demo rápida; remova o número para indexar tudo)
!python src/rag/build_index_chatbulario.py 8000


---
## 5. Subir a interface

Reaproveita a interface completa de `src/ui/gradio_app.py` (abas de consulta,
geração de PDF, auditoria e config).

- Primeiro "plugamos" o modelo real no cliente de LLM (senão os agentes usariam o mock).
- `inicializar_componentes()` já carrega o modelo agora (BioMistral base + adapter,
  ~5 min na 1ª vez). Pode aparecer um aviso vermelho de *timeout* na conversão de
  `safetensors` — **é inofensivo**, o modelo carrega mesmo assim.
- `demo.launch(share=True)` imprime um link público `*.gradio.live`.
  **Login:** usuário `medico` / senha `demo123`.


In [ ]:
import os, sys
os.chdir("/content/Techchalleng3")
sys.path.insert(0, "/content/Techchalleng3")
os.environ["GRADIO_ANALYTICS_ENABLED"] = "False"

# 1) plugar o modelo fine-tuned no singleton do LLM (usado por todos os agentes)
import src.llm.client as llmc
llmc._llm_instance = llmc.LLMClient(lora_path="/content/Techchalleng3/biomistral-medquad-lora")
print("LLM em modo:", "✅ REAL (fine-tuned)" if not llmc._llm_instance.use_mock else "⚠️ MOCK")

# 2) importar a app e pré-carregar componentes (LLM + RAG + gerador de PDF)
from src.ui.gradio_app import demo, inicializar_componentes
inicializar_componentes()

# 3) subir com link público
demo.launch(share=True, auth=("medico", "demo123"), show_error=True)


---
## (opcional) Teste rápido do pipeline, sem UI

Roda a cadeia **triagem → RAG → síntese → validação** direto no código, útil para
depurar sem depender do Gradio.


In [ ]:
from src.agents.triagem import triar
from src.agents.sintese import sintetizar
from src.agents.validacao import validar
from src.rag.retriever import Retriever

retriever = Retriever()
relato = "Homem 58 anos, dor torácica há 2h irradiando para o braço esquerdo, sudorese e falta de ar."

tri = triar(relato)
print("TRIAGEM:", tri)

rag_pmc     = retriever.retrieve_pmc(relato, k=4)
rag_interno = retriever.retrieve_interno(relato, k=4)
print("\nRAG interno:", [c["metadata"].get("nome_produto", c["source"]) for c in rag_interno])

sint = sintetizar(relato, rag_pmc, rag_interno)
final = validar(sint, tri, llmc._llm_instance)
import json
print("\nSÍNTESE FINAL:\n", json.dumps(final, indent=2, ensure_ascii=False)[:1500])
